# Qué añade XGBoost, explicado desde cero

Explicación de los cuatro puntos de la celda "What does XGBoost add?" de `XGBoost.ipynb`.
Primero el vocabulario que hace falta, con un ejemplo de una sola casa, y después los cuatro puntos.

# Parte 0: vocabulario

### Residuo
Es lo que le falta a la predicción para acertar:

```
residuo = valor real − predicción
```

Si una casa vale 200k y el modelo predice 150k, el residuo es +50k: la predicción se quedó corta en 50k. En el gradient boosting a mano, cada árbol nuevo intenta predecir estos residuos.

### Función de pérdida (o de error)
Es una fórmula que dice **cuánto de mal** está una predicción. La de este notebook es el **error cuadrático**:

```
error = ½ · (predicción − real)²
```

Se eleva al cuadrado para que fallar por arriba o por abajo cuente igual y para que fallar mucho cueste mucho más que fallar poco. El ½ solo está para que las cuentas de abajo salgan más limpias.

Con la casa de ejemplo (todo en miles):
- predicción 150 → error = ½ · (150 − 200)² = ½ · 2500 = **1250**
- predicción 151 → error = ½ · (151 − 200)² = ½ · 2401 = **1200,5**

### Derivada (aquí la llamamos **g**, de gradiente)
La derivada responde a esta pregunta: **si muevo la predicción un poquito, ¿cuánto cambia el error?** Es la **pendiente**, lo inclinada que está la cuesta en el punto donde estás.

En el ejemplo, al subir la predicción de 150 a 151 el error bajó unos 50 (de 1250 a 1200,5). La pendiente en 150 es aproximadamente **−50**.

Para el error cuadrático hay una fórmula exacta:

```
g = predicción − real
```

En el ejemplo, g = 150 − 200 = −50, que coincide con lo que vimos con números.

Lo importante del signo:
- **g negativo**: subir la predicción baja el error, así que hay que subir.
- **g positivo**: hay que bajar.
- **g = 0**: estás en el fondo y has acertado.

g es el residuo con el signo cambiado. El boosting a mano ya usaba esta información sin llamarla "derivada".

### Segunda derivada o curvatura (aquí la llamamos **h**)
Es la derivada de la derivada: **cuánto cambia la pendiente cuando me muevo**. Dice si la curva del error es muy curvada (la pendiente cambia rápido) o casi recta (cambia despacio).

En el ejemplo:

| predicción | pendiente g |
|---|---|
| 150 | −50 |
| 151 | −49 |
| 152 | −48 |

La pendiente sube **1** por cada unidad que avanzas, así que **h = 1**. Con el error cuadrático, h vale 1 siempre, en cualquier punto y para cualquier casa, porque la curva es una parábola perfecta.

### Descenso del gradiente vs. método de Newton
Son dos formas de decidir **cuánto mover** la predicción:

- **Descenso del gradiente**: usa solo g. "La cuesta baja hacia allí, doy un paso de tamaño fijo hacia allí." No sabe a qué distancia está el fondo.
- **Método de Newton**: usa g y h. Razona así: "La pendiente es −50 y sube 1 por cada unidad que avanzo. Para llegar a pendiente 0 (el fondo) tengo que avanzar 50."

  ```
  paso = −g / h = −(−50) / 1 = +50
  ```

  150 + 50 = 200, justo el valor real, en un solo salto.

# Parte 1: "Pasos que usan la curvatura del error"

Un árbol no da un valor distinto a cada casa: todas las casas que caen en la misma **hoja** (el final de una rama) reciben **el mismo valor**. Entonces XGBoost suma los g y los h de las casas de esa hoja:

- **G** = suma de los g de la hoja
- **H** = suma de los h de la hoja

Y aplica Newton a todo el grupo:

```
valor de la hoja = −G / (H + λ)
```

λ (lambda, una letra griega) es un ajuste que se explica en la parte 2. Por ahora vale 0.

**Ejemplo:** una hoja con 3 casas cuyos residuos son +10k, +20k y +30k.
- g de cada una: −10, −20, −30, así que G = −60
- h de cada una: 1, 1, 1, así que H = 3
- valor de la hoja = −(−60) / 3 = **20k**

20k es la **media de los residuos**, exactamente lo que hacía el boosting a mano. Con el error cuadrático, "usar la curvatura" da lo mismo que ya hacíamos.

**Entonces, ¿para qué sirve?** Para otras funciones de pérdida, como las de clasificación, cuya curva no es una parábola. Ahí h es distinto en cada fila, y Newton da pasos más cortos donde la curva es muy curvada y más largos donde es casi plana. Así XGBoost tiene una sola fórmula que funciona con cualquier función de pérdida.

(Al final, igual que en la versión a mano, ese valor se multiplica por `learning_rate`.)

# Parte 2: "Una penalización en el objetivo" (lambda y gamma)

Dos términos antes:
- **Objetivo**: lo que el modelo intenta hacer lo más pequeño posible. En la versión a mano era solo el error. En XGBoost es el error **más** unas penalizaciones.
- **Sobreajuste**: cuando el modelo memoriza las casas de entrenamiento (incluido su ruido) en vez de aprender patrones generales. Es lo que se ve en la gráfica de `XGBoost.ipynb`: el error de entrenamiento sigue bajando y el de validación no.

### λ (lambda, `reg_lambda`): encoge los valores de las hojas
En la fórmula, λ se **suma al denominador**, así que el resultado sale más pequeño. Misma hoja de 3 casas:

| | cuenta | valor de la hoja |
|---|---|---|
| Boosting a mano (λ = 0) | 60 / 3 | 20k |
| XGBoost con λ = 1 (el valor por defecto) | 60 / (3 + 1) | 15k |

Lo interesante es a quién afecta más:
- Hoja con **3 casas**: 60/3 = 20 frente a 60/4 = 15. Cambia mucho.
- Hoja con **1000 casas**: el denominador pasa de 1000 a 1001. Casi no cambia.

Es decir, **desconfía de las hojas con pocas casas**. Una media de 3 casas puede ser casualidad, una de 1000 no. Eso protege contra el sobreajuste.

### γ (gamma): un precio por cada hoja nueva
Al construir el árbol, en cada nodo el modelo se pregunta si vale la pena dividirlo en dos. Calcula la **ganancia**, que es cuánto baja el error si hace esa división, y solo divide si:

```
ganancia > γ
```

Los árboles a mano dividían siempre que `max_depth` lo permitiera. XGBoost añade una condición: la división tiene que mejorar lo bastante como para pagar su precio. Si no, esa rama se queda como hoja.

(`reg_alpha` es otra penalización parecida a lambda. Puede llevar a 0 las hojas que aportan poco.)

# Parte 3: "Valores faltantes con una dirección aprendida"

**NaN** ("Not a Number") es como pandas marca un dato que falta. En `XGBoost.ipynb`, `total_bedrooms` tiene 114.

Imagina que el árbol divide por `total_bedrooms < 500`. Las casas con dato van a la izquierda o a la derecha según su valor. Con las casas sin dato, XGBoost hace esto:
1. las manda todas a la izquierda y calcula la ganancia,
2. las manda todas a la derecha y calcula la ganancia,
3. se queda con el lado que dio más ganancia y lo guarda como **dirección por defecto** de esa división.

Al predecir una casa nueva sin `total_bedrooms`, sigue esa dirección guardada. Por eso el notebook funcionó **sin rellenar los huecos** (sin imputar con la mediana, como se hacía con el pipeline).

# Parte 4: "Velocidad"

Esto no cambia las predicciones, solo cuánto tarda:

- **Histogramas**: un histograma agrupa los valores en cajas, por ejemplo "de 0 a 100 dormitorios", "de 100 a 200"... La versión a mano probaba **todos** los umbrales posibles (un umbral es el número de corte, como el 500 de arriba). XGBoost primero mete cada columna en unas 256 cajas y solo prueba los bordes entre cajas. Son muchas menos pruebas y el resultado casi no empeora.
- **En paralelo**: busca la mejor división en varias columnas a la vez, usando varios núcleos del procesador.
- **GPU**: la tarjeta gráfica hace muchas cuentas pequeñas a la vez. Se activa con `device="cuda"`.

# Resumen

| Gradient boosting a mano | XGBoost |
|---|---|
| Hoja = media de los residuos | Hoja = suma de residuos / (casas + λ), así que las hojas pequeñas se encogen |
| Divide mientras la profundidad lo permita | Divide solo si la ganancia supera γ |
| Los NaN hay que rellenarlos antes | Los NaN van al lado que aprendió |
| Prueba todos los umbrales | Prueba los bordes de las cajas, en paralelo |